# Benchmark ASR zarma — test du modèle **LLM + `dje_Latn`** (Omnilingual)

Ce notebook fait tourner **`omniASR_LLM_300M_v2`** (qui respecte l'indication de langue,
contrairement au CTC) sur tes enregistrements, et produit un **dump d'hypothèses JSONL**
à rejouer localement dans le harnais du projet (`scripts/bench/run_benchmark.py --hypotheses`).

**Avant de lancer :** menu `Exécution` → `Modifier le type d'exécution` → **GPU (T4)**.

Ordre : 1) GPU+libs · 2) install · **3) REDÉMARRER la session** · 4) import · 5) upload · 6) transcription · 7) download.


## 1. Vérifier le GPU et installer les libs système


In [ ]:
!nvidia-smi -L || echo 'PAS DE GPU — active le GPU dans Exécution > Modifier le type d'exécution'
!apt-get -qq install -y libsndfile1 ffmpeg > /dev/null && echo 'libsndfile + ffmpeg OK'


## 2. Installer omnilingual-asr (aligner torchaudio + figer numpy)
Peut prendre quelques minutes (fairseq2, kenlm…). `omnilingual-asr` rétrograde numpy en 1.26.4 :
on le fige explicitement, puis **un redémarrage sera obligatoire** (cellule 3).


In [ ]:
# 1) omnilingual-asr 0.2.0 (--ignore-requires-python : la 0.2.0 borne à tort <=3.12 vs Python 3.12 de Colab)
!pip install -q --ignore-requires-python 'omnilingual-asr==0.2.0'
# 2) Forcer le TRIO cohérent depuis l'index cu128 (torch 2.8 <-> torchaudio 2.8 <-> torchvision 0.23)
#    -> évite les erreurs ABI 'torchaudio'/'torchvision::nms does not exist'
!pip install -q torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
# 3) Figer numpy 1.26.4 (compat fairseq2/numba) -> évite 'numpy.dtype size changed'
!pip install -q 'numpy==1.26.4'
print('Install OK -> Étape 3 : REDÉMARRER LA SESSION (cellule suivante).')


## 3. ⚠️ REDÉMARRER LA SESSION — étape obligatoire

Le rétrogradage de numpy impose un redémarrage du noyau pour recharger la bonne version.

**Fais : menu `Exécution` → `Redémarrer la session`.** (Les paquets installés restent — ne relance PAS les cellules 1 et 2.)

Puis reprends directement à la **cellule 4**.

> Astuce : la cellule ci-dessous force le redémarrage automatiquement. Après coupure, saute à la cellule 4.


In [ ]:
# Redémarrage PROPRE du noyau (évite le compteur 'restarting kernel (x/5)').
# La session va se déconnecter : c'est VOULU. Attends la reconnexion, puis va à l'ÉTAPE 4.
# (N'exécute cette cellule QU'UNE SEULE FOIS.)
try:
    from google.colab import runtime_manager  # API récente si dispo
    raise ImportError  # on privilégie la méthode ci-dessous, universelle
except Exception:
    pass
import IPython
print('Redémarrage du noyau… reprends ensuite à l’ÉTAPE 4 (ne relance ni 1, ni 2, ni 3).')
IPython.get_ipython().kernel.do_shutdown(restart=True)


## 4. Vérifier l'import + que le zarma est supporté
(à exécuter **après** le redémarrage)


In [ ]:
import torch, torchaudio, numpy
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
print('dje_Latn supporté :', 'dje_Latn' in supported_langs)


## 5. Uploader tes audios
Uploade **soit** un fichier **.zip** contenant tes `.wav`, **soit** plusieurs `.wav` directement.
Format attendu : **WAV mono 16 kHz** (comme ton dossier `v1`).


In [ ]:
import os, zipfile, glob, shutil
from google.colab import files
shutil.rmtree('audio', ignore_errors=True); os.makedirs('audio', exist_ok=True)
up = files.upload()   # sélectionne ton .zip (ou tes .wav)
for name in up:
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('audio')
    elif name.lower().endswith('.wav'):
        shutil.move(name, os.path.join('audio', os.path.basename(name)))
wavs = sorted(glob.glob('audio/**/*.wav', recursive=True))
print(f'{len(wavs)} fichier(s) .wav prêt(s)')
for w in wavs[:30]: print('  -', os.path.basename(w))


## 6. Charger le LLM et transcrire (avec `lang=["dje_Latn"]`)
Premier chargement = téléchargement des poids (~6,2 Go) : quelques minutes.


In [ ]:
# Cellule autonome : refait l'import + retrouve les fichiers (robuste après un redémarrage)
import time, torch, os, glob
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
wavs = sorted(glob.glob('audio/**/*.wav', recursive=True))
assert wavs, "Aucun .wav dans audio/ — relance la cellule d'upload."
print(len(wavs), 'fichier(s) à transcrire')

MODEL = 'omniASR_LLM_300M_v2'
try:
    pipe = ASRInferencePipeline(model_card=MODEL, device='cuda', dtype=torch.float16)
except Exception as e:
    print('float16 KO (', e, ') -> float32'); pipe = ASRInferencePipeline(model_card=MODEL, device='cuda', dtype=torch.float32)
print('modèle chargé')

records = []
print('\n=== TRANSCRIPTIONS LLM (lang=dje_Latn) ===')
for w in wavs:
    name = os.path.basename(w)
    t0 = time.perf_counter()
    txt = pipe.transcribe([w], lang=['dje_Latn'])[0]
    lat = int((time.perf_counter()-t0)*1000)
    records.append({'audio_path': name, 'text': txt, 'acoustic_score': 1.0,
                    'candidates': [], 'latency_ms': lat, 'model_version': MODEL})
    print(f'{name:12s} -> {txt!r}  ({lat} ms)')


## 7. Écrire le dump d'hypothèses et le télécharger
Récupère `hypotheses_llm.jsonl`, puis en local :
```bash
uv run python scripts/bench/run_benchmark.py \
    --manifest dataset/manifests/benchmark.jsonl \
    --hypotheses hypotheses_llm.jsonl --split test \
    --out docs/qa/benchmarks/benchmark-llm-test
```


In [ ]:
import json
with open('hypotheses_llm.jsonl','w',encoding='utf-8') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False, sort_keys=True)+'\n')
print('écrit hypotheses_llm.jsonl :', len(records), 'lignes')
from google.colab import files; files.download('hypotheses_llm.jsonl')


---
Ce notebook **mesure**, il n'invente rien. La vérité terrain (le vrai nombre de chaque fichier)
vient de **toi** ; le manifest se construit côté projet. Compare ensuite LLM vs CTC (story 5.4).
